In [ ]:
#r "nuget: Microsoft.DotNet.Interactive.SqlServer, *-*"

In [ ]:
#!connect mssql --kernel-name sql2025 --connection-string "Server=sql2025;TrustServerCertificate=True;Integrated Security=True"

In [ ]:
SELECT @@Version

In [ ]:
USE master
IF EXISTS (SELECT name FROM sys.databases WHERE name='ONNXDemo')BEGIN
    ALTER DATABASE ONNXDemo SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE ONNXDemo;
END
GO
CREATE DATABASE ONNXDemo
GO
USE ONNXDemo
GO
create master key encryption by password = 'MyTest!Mast3rP4ss'
GO
sp_configure 'external rest endpoint enabled', 1;
RECONFIGURE WITH OVERRIDE

In [ ]:
ALTER DATABASE SCOPED CONFIGURATION SET PREVIEW_FEATURES = ON

In [ ]:
EXEC sp_configure 'external AI runtimes enabled', 1
RECONFIGURE WITH OVERRIDE

In [ ]:
# Credits to Brian Spendolini
# https://devblogs.microsoft.com/azure-sql/create-embeddings-in-sql-server-2025-rc0-with-a-local-onnx-model-on-windows/

# Requires Machine Learning Services!

$OnnxPath = "C:\onnx_runtime"
$ModelPath=$OnnxPath + "\model"
$DLLPath=$OnnxPath + "\onnxruntime.dll"
$TokenizerPath=$OnnxPath + "\tokenizers_cpp.dll"
$ModelSource="https://huggingface.co/nsense/"
$ModelName="all-MiniLM-L6-v2-onnx"
if ((Test-Path $OnnxPath) -eq $false) { New-Item -Path $OnnxPath -ItemType Directory | Out-Null }
if ((Test-Path $ModelPath) -eq $false) { New-Item -Path $ModelPath -ItemType Directory | Out-Null }
if ((Test-Path $DLLPath) -eq $false) {
Invoke-WebRequest https://github.com/PARTHSQL/AIRuntimeDependencies/raw/refs/heads/main/bin/Release/onnxruntime.dll -OutFile $DLLPath
}
if ((Test-Path $TokenizerPath) -eq $false) {
Invoke-WebRequest https://github.com/PARTHSQL/tokenizers-cpp/releases/download/v0.1.1/tokenizers_cpp-release.dll -OutFile $TokenizerPath 
}
if ((Test-Path ($ModelPath + "\" + $ModelName)) -eq $false) {
cd $ModelPath
git clone ($ModelSource + "all-MiniLM-L6-v2-onnx")
}
$Acl = Get-Acl -Path $OnnxPath
$AccessRule = New-Object System.Security.AccessControl.FileSystemAccessRule("MSSQLLaunchpad", "FullControl", "ContainerInherit,ObjectInherit", "None","Allow")
# $Acl.AddAccessRule($AccessRule)
Set-Acl -Path $OnnxPath -AclObject $Acl
cd c:\

In [ ]:
CREATE EXTERNAL MODEL ONNXModel
WITH (
LOCATION = 'C:\onnx_runtime\model\all-MiniLM-L6-v2-onnx',
API_FORMAT = 'ONNX Runtime',
MODEL_TYPE = EMBEDDINGS,
MODEL = 'allMiniLM',
PARAMETERS = '{"valid":"JSON"}',
LOCAL_RUNTIME_PATH = 'C:\onnx_runtime\'
)

In [ ]:
SELECT ai_generate_embeddings (N'SEEEEEQUEL SEEEEEEERVER... 2025' USE MODEL ONNXModel)

In [ ]:
CREATE EXTERNAL MODEL ollamacpu
WITH (
    LOCATION = 'https://ai-cpu.lab.bwdemo.io:443/api/embed',
    API_FORMAT = 'ollama',
    MODEL_TYPE = EMBEDDINGS,
    MODEL = 'nomic-embed-text'
);

In [ ]:
SELECT ai_generate_embeddings (N'SEEEEEQUEL SEEEEEEERVER... 2025' USE MODEL ollamacpu)

In [ ]:
CREATE TABLE RandomTexts (
    RandomText NVARCHAR(max),
    embeddings VECTOR(768),
    embeddings_2 VECTOR(384)
);

In [ ]:
SET NOCOUNT ON;
DECLARE @response NVARCHAR(MAX);

EXEC sp_invoke_external_rest_endpoint
    @url = N'https://openlibrary.org/search.json?q=SQL%20Server',
    @method = 'GET',
    @response = @response OUTPUT;
 
TRUNCATE TABLE RandomTexts
INSERT INTO RandomTexts (RandomText)
SELECT Json_value(value, '$.title') +  isnull (' - ' + Json_value(value, '$.author_name[*]'),'')
FROM   Openjson(Json_query(@response, '$.result.docs'), 'strict $') AS export 

SET NOCOUNT OFF;


In [ ]:
SELECT TOP 5 RandomText FROM RandomTexts

In [ ]:
UPDATE RandomTexts SET embeddings = AI_GENERATE_EMBEDDINGS(RandomText USE MODEL ollamacpu)

In [ ]:
UPDATE RandomTexts SET embeddings_2 = AI_GENERATE_EMBEDDINGS(RandomText USE MODEL ONNXModel)